In [3]:
import pandas as pd
import numpy as np

In [4]:
# Load the dataset from the provided URL
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)

In [5]:
df.head() # display the first few rows of the dataset to get an overview of its structure and contents

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [6]:
df.shape # display the shape of the dataset to understand the number of rows and columns

(10000, 11)

# Question 1

In [7]:
df.isnull().sum() # check for missing values in each column of the dataset to identify any potential data quality issues

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

# Question 2

In [8]:
df["horsepower"].median() # calculate the median value of the "horsepower" column to use it for filling missing values

254.0

# Question 3

In [9]:
# Fill missing values in the "horsepower" column with the median value
df = df.copy()

np.random.seed(42) # set the random seed for reproducibility of the random operations that follow

n = len(df) # get the total number of samples in the dataset
n_val = int(n * 0.2) # calculate the number of samples for the validation set (20% of the total dataset)
n_test = int(n * 0.2) # calculate the number of samples for the test set (20% of the total dataset)
n_train = n - n_val - n_test # calculate the number of samples for the training set (the remaining 60% of the total dataset)

idx = np.arange(n) # create an array of indices from 0 to n-1 to represent the indices of the samples in the dataset
np.random.shuffle(idx) # shuffle the indices randomly to ensure that the samples are randomly distributed across the training, validation, and test sets

df_train = df.iloc[idx[:n_train]].copy() # create the training set by selecting the first n_train shuffled indices from the dataset and making a copy of the corresponding samples  
df_val = df.iloc[idx[n_train:n_train + n_val]].copy() # create the validation set by selecting the next n_val shuffled indices from the dataset and making a copy of the corresponding samples
df_test = df.iloc[idx[n_train + n_val:]].copy() # create the test set by selecting the remaining shuffled indices from the dataset and making a copy of the corresponding samples

In [10]:
len(df_train), len(df_val), len(df_test) # display the number of samples in the training, validation, and test sets to verify that they have been split correctly

(6000, 2000, 2000)

In [11]:
features = [
    "engine_displacement",
    "horsepower",
    "vehicle_weight",
    "model_year"
] # define the list of feature column names that will be used as input variables for the regression model

target = "fuel_efficiency_mpg" # define the target column name that will be used as the output variable for the regression model

In [12]:
def train_linear_regression(X, y):
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    
    return w # define a function to train a linear regression model using the normal equation method, which calculates the weights (coefficients) of the model based on the input features (X) and target variable (y)

In [13]:
def train_linear_regression(X, y):
    X = np.column_stack([np.ones(X.shape[0]), X])
    
    XTX = X.T @ X
    XTX_inv = np.linalg.inv(XTX)
    w = XTX_inv @ X.T @ y
    
    return w # define a function to train a linear regression model using the normal equation method, which calculates the weights (coefficients) of the model based on the input features (X) and target variable (y). The function also adds a bias term (intercept) to the input features by stacking a column of ones to the left of the feature matrix X.

In [14]:
def predict(X, w):
    X = np.column_stack([np.ones(X.shape[0]), X])
    return X @ w   # define a function to make predictions using the trained linear regression model. The function takes the input features (X) and the weights (w) as arguments, adds a bias term (intercept) to the input features, and returns the predicted values by performing a matrix multiplication of the modified input features and the weights.

In [15]:
def rmse(y, y_pred):
    error = y - y_pred
    mse = np.mean(error ** 2)
    return np.sqrt(mse)   # define a function to calculate the root mean squared error (RMSE) between the true target values (y) and the predicted values (y_pred). The function computes the error, calculates the mean squared error (MSE), and returns the square root of the MSE as the RMSE value.

In [16]:
X_train = df_train[features].fillna(0).values # fill missing values in the training set features with 0 and convert the DataFrame to a NumPy array for model training
y_train = df_train[target].values # convert the target variable in the training set to a NumPy array for model training

X_val = df_val[features].fillna(0).values # fill missing values in the validation set features with 0 and convert the DataFrame to a NumPy array for model evaluation
y_val = df_val[target].values # convert the target variable in the validation set to a NumPy array for model evaluation

w = train_linear_regression(X_train, y_train) # train the linear regression model using the training set features and target variable, and store the calculated weights (coefficients) in the variable w

y_pred = predict(X_val, w) # make predictions on the validation set features using the trained linear regression model and the calculated weights (coefficients), and store the predicted values in the variable y_pred

rmse(y_val, y_pred) # calculate the root mean squared error (RMSE) between the true target values in the validation set and the predicted values, and display the RMSE value to evaluate the performance of the trained linear regression model on the validation set

np.float64(2.2052931947534096)

In [17]:
mean_hp = df_train["horsepower"].mean()

mean_hp # display the mean value of the "horsepower" column in the training set to understand the average horsepower of the vehicles in the dataset

np.float64(254.46338337605272)

In [18]:
X_train = df_train[features].fillna(mean_hp).values # fill missing values in the training set features with the mean value of the "horsepower" column and convert the DataFrame to a NumPy array for model training
y_train = df_train[target].values # convert the target variable in the training set to a NumPy array for model training
X_val = df_val[features].fillna(mean_hp).values # fill missing values in the validation set features with the mean value of the "horsepower" column and convert the DataFrame to a NumPy array for model evaluation

w = train_linear_regression(X_train, y_train) # train the linear regression model using the training set features and target variable, and store the calculated weights (coefficients) in the variable w

y_pred = predict(X_val, w) # make predictions on the validation set features using the trained linear regression model and the calculated weights (coefficients), and store the predicted values in the variable y_pred

rmse(y_val, y_pred) # calculate the root mean squared error (RMSE) between the true target values in the validation set and the predicted values, and display the RMSE value to evaluate the performance of the trained linear regression model on the validation set after filling missing values with the mean horsepower.

np.float64(2.2018174845674756)

# Question 4

In [19]:
# define a function to train a linear regression model with L2 regularization (Ridge Regression) using the normal equation method. The function takes the input features (X), target variable (y), and regularization parameter (r) as arguments, adds a bias term (intercept) to the input features, and calculates the weights (coefficients) of the model based on the modified input features and target variable. The regularization term is added to the XTX matrix to prevent overfitting and improve generalization of the model.


def train_linear_regression_reg(X, y, r):
    X = np.column_stack([np.ones(X.shape[0]), X]) # add a bias term (intercept) to the input features by stacking a column of ones to the left of the feature matrix X, which allows the model to learn an intercept term in addition to the weights for the input features.
     
    XTX = X.T @ X # calculate the dot product of the transposed feature matrix (X.T) and the feature matrix (X) to obtain the XTX matrix, which is used in the normal equation for linear regression
     
    XTX = XTX + r * np.eye(XTX.shape[0]) # add the regularization term (r * np.eye(XTX.shape[0])) to the XTX matrix to prevent overfitting and improve generalization of the model. The regularization term is a diagonal matrix with the regularization parameter (r) on the diagonal, which penalizes large weights in the model and encourages smaller weights for better generalization.
    
    XTX_inv = np.linalg.inv(XTX) # calculate the inverse of the XTX matrix using NumPy's linear algebra module (np.linalg.inv) to obtain the XTX_inv matrix, which is used in the normal equation for linear regression with regularization. The regularization term (r * np.eye(XTX.shape[0])) is added to the XTX matrix to prevent overfitting and improve generalization of the model.

    w = XTX_inv @ X.T @ y # calculate the weights (coefficients) of the linear regression model using the normal equation method, which involves multiplying the inverse of the XTX matrix with the transposed feature matrix (X.T) and the target variable (y) to obtain the optimal weights that minimize the mean squared error between the predicted and actual target values.
    
    return w 

In [20]:
X_train = df_train[features].fillna(0).values # fill missing values in the training set features with 0 and convert the DataFrame to a NumPy array for model training
X_val = df_val[features].fillna(0).values # fill missing values in the validation set features with 0 and convert the DataFrame to a NumPy array for model evaluation

results = {} # create an empty dictionary to store the RMSE scores for different regularization parameters (r) when training the linear regression model with L2 regularization (Ridge Regression)

for r in [0, 0.01, 0.1, 1, 5, 10, 100]: # iterate over a list of regularization parameters (r) to train the linear regression model with L2 regularization (Ridge Regression) for each value of r, make predictions on the validation set, calculate the RMSE score, and store the results in the dictionary for comparison of model performance across different regularization strengths.
    
    w = train_linear_regression_reg(X_train, y_train, r) # train the linear regression model with L2 regularization (Ridge Regression) using the training set features and target variable for the current value of the regularization parameter (r), and store the calculated weights (coefficients) in the variable w

    y_pred = predict(X_val, w) # make predictions on the validation set features using the trained linear regression model with L2 regularization (Ridge Regression) and the calculated weights (coefficients), and store the predicted values in the variable y_pred
    
    score = rmse(y_val, y_pred) # calculate the root mean squared error (RMSE) between the true target values in the validation set and the predicted values for the current value of the regularization parameter (r), and store the RMSE score in the variable score to evaluate the performance of the trained linear regression model with L2 regularization (Ridge Regression) on the validation set.
    results[r] = round(score, 4) # store the RMSE score for the current value of the regularization parameter (r) in the results dictionary, rounding the score to 4 decimal places for better readability and comparison of model performance across different regularization strengths.

results

{0: np.float64(2.2053),
 0.01: np.float64(2.2058),
 0.1: np.float64(2.2241),
 1: np.float64(2.3492),
 5: np.float64(2.4094),
 10: np.float64(2.4195),
 100: np.float64(2.4292)}

# Question 5

In [ ]:
scores = [] # create an empty list to store the RMSE scores for different random seeds when training the linear regression model with L2 regularization (Ridge Regression) and evaluating its performance on the validation set. This allows for assessing the stability and robustness of the model's performance across different random splits of the dataset.

for seed in range(10): # iterate over a range of random seeds (0 to 9) to train the linear regression model with L2 regularization (Ridge Regression) and evaluate its performance on the validation set for each random seed. This allows for assessing the stability and robustness of the model's performance across different random splits of the dataset.
    
    np.random.seed(seed) # set the random seed for reproducibility of the random operations that follow, ensuring that the same random splits of the dataset are generated for each iteration of the loop. This allows for consistent evaluation of the model's performance across different random seeds.
    
    idx = np.arange(n) # create an array of indices from 0 to n-1 to represent the indices of the samples in the dataset for the current random seed iteration
    np.random.shuffle(idx) # shuffle the indices randomly to ensure that the samples are randomly distributed across the training, validation, and test sets for the current random seed iteration
    
    df_train = df.iloc[idx[:n_train]] # create the training set by selecting the first n_train shuffled indices from the dataset for the current random seed iteration
    df_val = df.iloc[idx[n_train:n_train + n_val]] # create the validation set by selecting the next n_val shuffled indices from the dataset for the current random seed iteration
    
    X_train = df_train[features].fillna(0).values # fill missing values in the training set features with 0 and convert the DataFrame to a NumPy array for model training for the current random seed iteration
    y_train = df_train[target].values # convert the target variable in the training set to a NumPy array for model training for the current random seed iteration
    
    X_val = df_val[features].fillna(0).values # fill missing values in the validation set features with 0 and convert the DataFrame to a NumPy array for model evaluation for the current random seed iteration
    y_val = df_val[target].values # convert the target variable in the validation set to a NumPy array for model evaluation for the current random seed iteration
    
    w = train_linear_regression(X_train, y_train) # train the linear regression model using the training set features and target variable for the current random seed iteration, and store the calculated weights (coefficients) in the variable w
    
    y_pred = predict(X_val, w) # make predictions on the validation set features using the trained linear regression model and the calculated weights (coefficients) for the current random seed iteration, and store the predicted values in the variable y_pred
    
    score = rmse(y_val, y_pred) # calculate the root mean squared error (RMSE) between the true target values in the validation set and the predicted values for the current random seed iteration, and store the RMSE score in the variable score to evaluate the performance of the trained linear regression model on the validation set.
    scores.append(score) # append the RMSE score for the current random seed iteration to the scores list, allowing for the collection of RMSE scores across different random seeds to assess the stability and robustness of the model's performance on the validation set.

scores

[np.float64(2.2392041430461265),
 np.float64(2.202112821083884),
 np.float64(2.1634197787530947),
 np.float64(2.2043588768539215),
 np.float64(2.2018800943311554),
 np.float64(2.2457851509085134),
 np.float64(2.2697212079524043),
 np.float64(2.190794599933443),
 np.float64(2.216611201686444),
 np.float64(2.207036592817896)]

In [ ]:
np.std(scores) # calculate the standard deviation of the RMSE scores collected across different random seeds to assess the variability and stability of the linear regression model's performance on the validation set. A lower standard deviation indicates more consistent performance across different random splits of the dataset, while a higher standard deviation suggests greater variability in the model's performance.

np.float64(0.02878127407819125)

# Question 6

In [ ]:
# define a function to train a linear regression model with L2 regularization (Ridge Regression) using the normal equation method. The function takes the input features (X), target variable (y), and regularization parameter (r) as arguments, adds a bias term (intercept) to the input features, and calculates the weights (coefficients) of the model based on the modified input features and target variable. The regularization term is added to the XTX matrix to prevent overfitting and improve generalization of the model.

np.random.seed(9) # set the random seed to 9 for reproducibility of the random operations that follow, ensuring that the same random splits of the dataset are generated for the subsequent training, validation, and test sets.

idx = np.arange(n) # create an array of indices from 0 to n-1 to represent the indices of the samples in the dataset for the current random seed iteration
np.random.shuffle(idx) # shuffle the indices randomly to ensure that the samples are randomly distributed across the training, validation, and test sets for the current random seed iteration

df_train = df.iloc[idx[:n_train]] # create the training set by selecting the first n_train shuffled indices from the dataset and making a copy of the corresponding samples
df_val = df.iloc[idx[n_train:n_train + n_val]] # create the validation set by selecting the next n_val shuffled indices from the dataset and making a copy of the corresponding samples
df_test = df.iloc[idx[n_train + n_val:]] # create the test set by selecting the remaining shuffled indices from the dataset and making a copy of the corresponding samples


In [ ]:
df_full_train = pd.concat([df_train, df_val]) # concatenate the training and validation sets to create a full training set (df_full_train) that combines the samples from both sets, allowing for training the linear regression model on a larger dataset before evaluating its performance on the test set.

In [ ]:
X_full_train = df_full_train[features].fillna(0).values # fill missing values in the full training set features with 0 and convert the DataFrame to a NumPy array for model training
y_full_train = df_full_train[target].values # convert the target variable in the full training set to a NumPy array for model training

X_test = df_test[features].fillna(0).values # fill missing values in the test set features with 0 and convert the DataFrame to a NumPy array for model evaluation
y_test = df_test[target].values # convert the target variable in the test set to a NumPy array for model evaluation

In [ ]:
# define a function to train a linear regression model with L2 regularization (Ridge Regression) using the normal equation method. The function takes the input features (X), target variable (y), and regularization parameter (r) as arguments, adds a bias term (intercept) to the input features, and calculates the weights (coefficients) of the model based on the modified input features and target variable. The regularization term is added to the XTX matrix to prevent overfitting and improve generalization of the model.
w = train_linear_regression_reg(
    X_full_train,
    y_full_train,
    r=0.001
)

In [ ]:
y_pred = predict(X_test, w) # make predictions on the test set features using the trained linear regression model with L2 regularization (Ridge Regression) and the calculated weights (coefficients), and store the predicted values in the variable y_pred

rmse(y_test, y_pred) # calculate the root mean squared error (RMSE) between the true target values in the test set and the predicted values, and display the RMSE value to evaluate the performance of the trained linear regression model with L2 regularization (Ridge Regression) on the test set.

np.float64(2.235827747191731)